# RoboImagine: released checkpoint → predicted video
Install `pip install -e ".[notebook]"` in this kernel first. Run All downloads one released dataset/variant plus pinned T5. CUDA recommended. No training or paid API calls.

In [ ]:
from pathlib import Path
import sys, subprocess, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
sys.path.insert(0, str(root/'src'))
dataset = 'rt1'  # rt1 / bridge / robocasa
variant = 'main'  # main / sft
example_index = 0
output = root/'predictions'/f'{dataset}_{variant}_{example_index:03d}'
# Optional offline inputs; leave None for public Hugging Face download.
local_bundle = None
local_example = None
local_reference = None
local_text_model = None

In [ ]:
from roboimagine.hub import download_bundle
if local_bundle is None:
    bundle, example, reference = download_bundle(dataset, variant, root/'weights', example_index=example_index)
else:
    assert local_example is not None
    bundle, example, reference = Path(local_bundle), Path(local_example), local_reference
print('Selected checkpoint and example; the CLI verifies bundle hashes before inference.')

In [ ]:
import os
env = os.environ.copy()
env['PYTHONPATH'] = str(root/'src') + os.pathsep + env.get('PYTHONPATH','')
command = [sys.executable, '-m', 'roboimagine.cli', '--dataset', dataset, '--variant', variant, '--bundle', str(bundle), '--input', str(example), '--output', str(output)]
if variant == 'main' and reference is not None:
    command += ['--reference', str(reference)]
if local_text_model is not None:
    command += ['--text-model', str(local_text_model)]
subprocess.run(command, env=env, check=True)
report = json.loads((output/'run.json').read_text())
report

In [ ]:
from IPython.display import Video, display
display(Video(str(output/'prediction.mp4'), embed=True))
if (output/'ground_truth.mp4').exists():
    display(Video(str(output/'ground_truth.mp4'), embed=True))

The reference check compares this saved example with the archived prediction under the released main checkpoint. Hardware/kernel changes can affect numerical equality. Demo scores and timings are not the paper's full-cohort means or warmed-up timing protocol. See `docs/INFERENCE.md` and `experiments/README.md`.